In [ ]:
from huggingface_hub import login
import gc
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, BitsAndBytesConfig
from langdetect import detect

print(f"CUDA System Acceleration Detected: {torch.cuda.is_available()}")
api_token = input("Enter your Access Token: ")
login(api_token)

In [ ]:
class NotebookTranslator:
    def __init__(self):
        print("Loading Gemma Engine...")
        self.model_name = "google/gemma-2-2b-it"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        
        if torch.cuda.is_available():
            self.device = torch.device("cuda")
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=quant_config,
                device_map="auto"
            )
            print(f"Hardware Target Locked: GPU ({torch.cuda.get_device_name(0)}) with 4-Bit Optimization")
        else:
            self.device = torch.device("cpu")
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
                device_map=None,
                low_cpu_mem_usage=True
            ).to(self.device)
            print("Hardware Target Locked: CPU (High-Speed Native RAM Mode)")
            
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        self.LANG_MAP = {
            "en": "English", 
            "fr": "French", 
            "es": "Spanish", 
            "de": "German", 
            "da": "Danish", 
            "zh": "Chinese"
        }
                
    def _flush_memory(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def detect_language(self, text):
        try:
            return detect(text) if (text and any(c.isalpha() for c in text)) else "en"
        except Exception:
            return "en"
            
    def translate(self, text, source, target):
        self._flush_memory()
        tgt_lang = self.LANG_MAP.get(target, target)
        
        prompt_transl = (
            f"Translate the following text strictly into {tgt_lang}. "
            f"Provide only the direct translation output with no introductory text, notes, or explanations.\n\n"
            f"Text to translate:\n\"{text}\""
        )
        
        # FIXED: Replaced llama-cpp logic with native Transformers chat formatting and generation loops
        messages = [{"role": "user", "content": prompt_transl}]
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            generated_tokens = self.model.generate(
                **inputs,
                max_new_tokens=350,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id
            )
            
        input_length = inputs.input_ids.shape[1]
        raw_output_text = self.tokenizer.decode(generated_tokens[0][input_length:], skip_special_tokens=True)
        return raw_output_text.strip().strip('"')

    def generate_ai_response(self, prompt_text, max_new_tokens=550):
        self._flush_memory()
        messages = [
            {"role": "system", "content": "You are a helpful, direct assistant. Answer the user's questions accurately and concisely without any fluff."},
            {"role": "user", "content": prompt_text}
        ]
        
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            generated_tokens = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id
            )
            
        input_length = inputs.input_ids.shape[1]
        response_text = self.tokenizer.decode(generated_tokens[0][input_length:], skip_special_tokens=True)
        return response_text.strip()
    
# Instantiate the translator module
translator = NotebookTranslator()
print("Multilingual Engine ready and optimized for speed!")

In [ ]:
def extract_questions(text):
    return re.findall(r'\d+\..*?\?', text) or [text]

In [ ]:
print("\n" + "="*40)
enquiry_text = input("Enter your enquiry: ").strip()  
# Added target language preference input
target_pref = input("Enter desired target language code (e.g., 'en', 'fr', 'es', 'de'): ").strip().lower()
print("="*40)

In [ ]:
# DETECT SOURCE LANGUAGE 
detect_lang = translator.detect_language(enquiry_text)
translated_enq = translator.translate(enquiry_text, source=detect_lang, target=target_pref)

# Print results
print(f"Original Enquiry: {enquiry_text}")
print(f"Detected Language: {detect_lang.upper()}")
print(f"Translated Response Target ({target_pref.upper()}): {translated_enq}")

In [ ]:
# QUESTION SEARCH
def contains_question(text):
    return '?' in text.strip()

print(f"Contains Question? {contains_question(enquiry_text)}")

In [ ]:
# Response in chosen tongue
ai_response = translator.generate_ai_response(enquiry_text) 
print(f"AI Response:\n{ai_response}\n")

# Re-translated response in user's preferred language
response_transl = translator.translate(ai_response, target=target_pref)
print(f"Translated Response ({target_pref.upper()}):\n{response_transl}")